# 03 - 迁移学习

## 学习目标
- 理解迁移学习的原理
- 使用预训练模型
- Fine-tuning 实战

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
import matplotlib.pyplot as plt
import numpy as np

## 1. 加载预训练模型

In [ ]:
# 加载 VGG16（不含顶层）
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# 冻结基础模型
base_model.trainable = False

print(f'基础模型层数: {len(base_model.layers)}')
print(f'可训练参数: {base_model.count_params()}')

## 2. 构建完整模型

In [ ]:
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')  # 10 类分类
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 3. 使用 CIFAR-10 演示

In [ ]:
# 加载数据
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# 调整大小到 224x224
x_train = tf.image.resize(x_train, [224, 224]).numpy()
x_test = tf.image.resize(x_test, [224, 224]).numpy()

# 预处理
x_train = tf.keras.applications.vgg16.preprocess_input(x_train)
x_test = tf.keras.applications.vgg16.preprocess_input(x_test)

print(f'训练集: {x_train.shape}')
print(f'测试集: {x_test.shape}')

## 4. 训练（第一阶段：只训练顶层）

In [ ]:
# 只训练新添加的层
history1 = model.fit(
    x_train[:1000], y_train[:1000],  # 使用子集演示
    validation_data=(x_test[:200], y_test[:200]),
    epochs=5,
    batch_size=32
)

print(f'第一阶段准确率: {history1.history["val_accuracy"][-1]:.4f}')

## 5. Fine-tuning（第二阶段）

In [ ]:
# 解冻部分层
base_model.trainable = True

# 冻结前面的层，只训练后面的层
for layer in base_model.layers[:15]:
    layer.trainable = False

# 使用较小的学习率
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 继续训练
history2 = model.fit(
    x_train[:1000], y_train[:1000],
    validation_data=(x_test[:200], y_test[:200]),
    epochs=5,
    batch_size=32
)

print(f'Fine-tuning 后准确率: {history2.history["val_accuracy"][-1]:.4f}')

## 小结
- 迁移学习利用预训练知识
- 先冻结基础模型，再逐步解冻
- Fine-tuning 使用小学习率